# AutoReCoder — Migration Progress Analysis

Equivalent to autoresearch's analysis notebook.
Visualizes the migration progress from `results/results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

RESULTS_PATH = Path('../results/results.tsv')
OUTPUT_DIR = Path('../results')

In [ ]:
# Phase 5.1 — Load results.tsv

df = pd.read_csv(RESULTS_PATH, sep='\t')
print(f'Loaded {len(df)} experiments')
df.head()

In [ ]:
# Classify experiments
fail_mask = df['compile_status'] == 'fail'
keep_mask = (~fail_mask) & (df['unsafe_after'] < df['unsafe_before'])
discard_mask = (~fail_mask) & (~keep_mask)

keep_df    = df[keep_mask].copy()
discard_df = df[discard_mask].copy()
fail_df    = df[fail_mask].copy()

print(f'Kept:             {keep_mask.sum()} ({100*keep_mask.mean():.1f}%)')
print(f'Discarded:        {discard_mask.sum()} ({100*discard_mask.mean():.1f}%)')
print(f'Compile failures: {fail_mask.sum()} ({100*fail_mask.mean():.1f}%)')

# Compute cumulative best frontier (running min of unsafe_count over time)
if 'unsafe_after' in df.columns and len(keep_df) > 0:
    # Build a series of (experiment_index, unsafe_count) for kept experiments
    frontier_x = [0]
    frontier_y = [df['unsafe_before'].iloc[0] if len(df) > 0 else 0]
    current_best = frontier_y[0]
    for idx, row in df.iterrows():
        exp_num = idx + 1
        if keep_mask[idx]:
            current_best = min(current_best, row['unsafe_after'])
        frontier_x.append(exp_num)
        frontier_y.append(current_best)
else:
    frontier_x, frontier_y = [], []

In [ ]:
# Phase 5.2 — Progress Curve

fig, ax = plt.subplots(figsize=(14, 6))

exp_nums = np.arange(1, len(df) + 1)

# Compile failures: orange triangles
if fail_mask.any():
    fail_x = exp_nums[fail_mask.values]
    fail_y = df.loc[fail_mask, 'unsafe_before'].values
    ax.scatter(fail_x, fail_y, marker='^', color='orange', s=80,
               zorder=3, label='Compile failure')

# Discarded: red X
if discard_mask.any():
    disc_x = exp_nums[discard_mask.values]
    disc_y = df.loc[discard_mask, 'unsafe_before'].values
    ax.scatter(disc_x, disc_y, marker='x', color='red', s=80,
               zorder=3, label='Discarded')

# Kept: green dots
if keep_mask.any():
    keep_x = exp_nums[keep_mask.values]
    keep_y = df.loc[keep_mask, 'unsafe_after'].values
    ax.scatter(keep_x, keep_y, marker='o', color='green', s=80,
               zorder=4, label='Kept')
    # Annotate with function name (for kept experiments)
    for x, y, row in zip(keep_x, keep_y, keep_df.itertuples()):
        ax.annotate(row.function_name, (x, y), fontsize=6,
                    xytext=(4, 4), textcoords='offset points', alpha=0.7)

# Frontier line: blue
if frontier_x:
    ax.step(frontier_x, frontier_y, where='post', color='steelblue',
            linewidth=2, label='Best unsafe_count', zorder=2)

# Goal line
ax.axhline(0, color='purple', linestyle='--', linewidth=1, alpha=0.5, label='Goal (unsafe=0)')

ax.set_xlabel('Experiment number')
ax.set_ylabel('unsafe_count')
ax.set_title('AutoReCoder Migration Progress')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = OUTPUT_DIR / 'progress.png'
plt.savefig(out_path, dpi=150)
print(f'Saved → {out_path}')
plt.show()

In [ ]:
# Phase 5.3 — Pattern Effectiveness Table

if len(df) > 0 and 'pattern_used' in df.columns:
    pattern_stats = []
    for pattern, group in df.groupby('pattern_used'):
        attempts = len(group)
        successes = (group['unsafe_after'] < group['unsafe_before']).sum()
        success_rate = successes / attempts if attempts > 0 else 0.0
        reductions = (group['unsafe_before'] - group['unsafe_after'])
        avg_reduction = reductions[reductions > 0].mean() if successes > 0 else 0.0
        pattern_stats.append({
            'pattern': pattern,
            'attempts': attempts,
            'successes': successes,
            'success_rate': success_rate,
            'avg_unsafe_reduction': avg_reduction,
        })

    pattern_df = pd.DataFrame(pattern_stats).sort_values('success_rate', ascending=False)
    pattern_df['success_rate'] = pattern_df['success_rate'].map('{:.1%}'.format)
    pattern_df['avg_unsafe_reduction'] = pattern_df['avg_unsafe_reduction'].map('{:.1f}'.format)
    print(pattern_df.to_string(index=False))
else:
    print('No data yet — run some experiments first.')

In [ ]:
# Phase 5.4 — Summary Statistics

if len(df) == 0:
    print('No experiments logged yet.')
else:
    starting = df['unsafe_before'].iloc[0]
    current  = keep_df['unsafe_after'].iloc[-1] if len(keep_df) > 0 else starting
    total    = len(df)
    n_kept   = keep_mask.sum()
    n_disc   = discard_mask.sum()
    n_fail   = fail_mask.sum()
    net_pct  = 100 * (starting - current) / starting if starting > 0 else 0.0

    print(f'Starting unsafe_count:  {starting}')
    print(f'Current unsafe_count:   {current}')
    print(f'Total experiments:      {total}')
    print(f'Kept:                   {n_kept} ({100*n_kept/total:.1f}%)')
    print(f'Discarded:              {n_disc} ({100*n_disc/total:.1f}%)')
    print(f'Compile failures:       {n_fail} ({100*n_fail/total:.1f}%)')
    print(f'Net improvement:        {net_pct:.1f}%')